# Examen final — Sistema RAG sobre arXiv Paper Abstracts

**Asignatura:** Recuperación de Información  
**Corpus:** [arXiv Paper Abstracts — Kaggle](https://www.kaggle.com/datasets/spsayakpaul/arxiv-paper-abstracts)

Este notebook documenta un sistema completo que descarga y prepara el corpus, genera embeddings, persiste un índice vectorial, recupera y reordena evidencias, genera respuestas fundamentadas y las presenta en un chat web desplegable.

> **Alcance de evaluación:** se documenta una evaluación cualitativa de la respuesta y de sus evidencias. No se incluyen métricas estándar de recuperación, de acuerdo con el alcance solicitado para esta implementación.


## 0. Configuración

Las claves se leen exclusivamente desde variables de entorno o desde un archivo local `.env` excluido de Git. Antes de ejecutar, instalar `requirements.txt` y definir `LLM_API_KEY` para habilitar la generación.


In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Ejecute el notebook desde la raíz del proyecto")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display

from arxiv_rag.config import Settings

settings = Settings.from_env()
print("Directorio de datos:", settings.data_dir)
print("Modelo de embeddings:", settings.embedding_model)
print("LLM configurado:", bool(settings.llm_api_key))


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## A. Preparación del corpus

Se descarga el dataset público mediante `kagglehub`. La implementación elige la variante actualizada con columnas `terms`, `titles` y `abstracts`; normaliza espacios, elimina registros vacíos, deduplica por título y abstract, combina categorías repetidas y genera identificadores estables. Los abstracts se fragmentan en ventanas solapadas para conservar contexto.


In [ ]:
from arxiv_rag.corpus import download_corpus, iter_chunks, load_corpus, select_corpus_file

dataset_dir = download_corpus()
csv_path = select_corpus_file(dataset_dir)
documents = load_corpus(csv_path)  # corpus completo deduplicado
chunks = list(iter_chunks(documents, settings.chunk_size, settings.chunk_overlap))

corpus_summary = pd.DataFrame({
    "archivo": [csv_path.name],
    "documentos_limpios": [len(documents)],
    "fragmentos": [len(chunks)],
    "categorias_distintas": [len({c for d in documents for c in d.categories})],
})
display(corpus_summary)
display(pd.DataFrame([
    {"doc_id": d.doc_id, "title": d.title, "categories": ", ".join(d.categories), "abstract": d.abstract[:300]}
    for d in documents[:3]
]))


## B. Representación mediante embeddings

Cada representación combina título, categorías y fragmento del abstract. Se usa `sentence-transformers/all-MiniLM-L6-v2`; los vectores se normalizan para que el producto interno corresponda a similitud coseno.


In [ ]:
from arxiv_rag.embeddings import EmbeddingService

embedder = EmbeddingService(settings.embedding_model)
sample_vectors = embedder.encode([chunk.embedding_text for chunk in chunks[:3]])
print("Forma de la matriz:", sample_vectors.shape)
print("Normas L2:", (sample_vectors ** 2).sum(axis=1) ** 0.5)


## C. Almacenamiento y búsqueda vectorial

Chroma persiste documentos, metadatos y embeddings en `data/chroma`. La colección usa espacio coseno y un índice HNSW. El manifiesto registra fuente, modelo y parámetros de fragmentación para garantizar trazabilidad.

La primera ejecución procesa el corpus completo y puede tardar varios minutos en CPU. Si ya existe un índice compatible, se reutiliza; use `reset=True` solo cuando sea necesario reconstruirlo.


In [ ]:
from arxiv_rag.indexing import build_index
from arxiv_rag.vector_store import VectorStore

index_stats = build_index(settings, csv_path=csv_path, max_documents=None, reset=False)
store = VectorStore(settings.chroma_dir, settings.collection_name, settings.embedding_model)
print(index_stats)
print("Fragmentos persistidos en Chroma:", store.count)


## D. Recuperación y re-ranking

La consulta se codifica con el mismo modelo. Chroma devuelve 20 candidatos semánticos y `cross-encoder/ms-marco-MiniLM-L-6-v2` los reordena evaluando conjuntamente consulta y fragmento. Se conservan cinco evidencias finales y, por defecto, un fragmento por paper para aumentar la diversidad documental.


In [ ]:
from arxiv_rag.retriever import Retriever

example_query = "What are the main applications of Graph Neural Networks?"
retriever = Retriever(settings)
retrieval = retriever.retrieve(example_query)

display(pd.DataFrame([e.to_dict() for e in retrieval.evidence])[
    ["evidence_id", "title", "categories", "semantic_score", "rerank_score"]
])
print("Contexto insuficiente:", retrieval.insufficient)
print(f"Tiempo de recuperación: {retrieval.retrieval_ms / 1000:.2f} s")


## E. Generación aumentada por recuperación

El generador recibe exclusivamente las evidencias seleccionadas. El prompt exige responder en el idioma de la consulta, citar cada afirmación con etiquetas `[E#]`, no obedecer instrucciones presentes en el corpus y declarar falta de contexto. Además, el pipeline evita invocar el LLM cuando los puntajes de recuperación quedan bajo los umbrales de abstención.


In [ ]:
from arxiv_rag.rag import RAGPipeline

rag = RAGPipeline(settings)
if settings.llm_api_key:
    result = rag.answer(example_query)
    display(Markdown(result.answer))
    print("¿Respuesta insuficiente?", result.insufficient)
else:
    print("Defina LLM_API_KEY en .env para ejecutar la generación. La recuperación anterior funciona sin esa clave.")


## F. Presentación de evidencias

Cada evidencia conserva un identificador que coincide con las citas de la respuesta, título del paper, categorías, fragmento exacto y puntuaciones de ambas etapas. Esto permite contrastar directamente la consulta, la afirmación generada y su fuente.


In [ ]:
def evidence_table(items):
    return pd.DataFrame([
        {
            "cita": f"[{e.evidence_id}]",
            "paper": e.title,
            "categorías": ", ".join(e.categories),
            "similitud": round(e.semantic_score, 4),
            "re-ranking": None if e.rerank_score is None else round(e.rerank_score, 4),
            "fragmento": e.text,
        }
        for e in items
    ])

items_to_show = result.evidence if settings.llm_api_key else retrieval.evidence
display(evidence_table(items_to_show))


## G. Interfaz web conversacional

`app.py` implementa un chat Streamlit. Permite enviar múltiples preguntas, muestra la respuesta, expande las evidencias y reporta los tiempos de recuperación/generación. Aunque la sesión muestra el historial, cada pregunta se procesa de forma independiente y no se incluye el diálogo anterior en el prompt.


In [ ]:
APP_COMMAND = "streamlit run app.py"
print("Ejecute en una terminal desde la raíz del proyecto:")
print(APP_COMMAND)
print("URL local: http://localhost:7860")
# Para lanzarlo desde Jupyter, descomente la siguiente línea:
# !streamlit run app.py


## H. Despliegue en la nube

El repositorio incluye `Dockerfile` y metadatos para un Hugging Face Space con SDK Docker. En el servicio se debe crear `LLM_API_KEY` como **secreto**, nunca como parte del código. `AUTO_BUILD_INDEX=true` permite descargar e indexar el corpus en el primer inicio. La URL definitiva debe reemplazar el marcador siguiente antes de entregar.


In [ ]:
DEPLOYED_URL = os.getenv(
    "DEPLOYED_URL",
    "https://huggingface.co/spaces/USUARIO/NOMBRE-DEL-SPACE",  # reemplazar
)
if "USUARIO" in DEPLOYED_URL:
    print("⚠ Pendiente: reemplace DEPLOYED_URL con la URL pública de la aplicación.")
print("URL del chat RAG:", DEPLOYED_URL)


## I. Evaluación cualitativa del sistema y de la generación

La revisión se realiza manualmente considerando: corrección, relevancia, fidelidad a las evidencias, integración de varios documentos y reconocimiento de información insuficiente. Se recomienda justificar cada juicio y conservar ejemplos de errores. La última consulta siguiente está deliberadamente fuera del dominio para comprobar la abstención.


In [ ]:
evaluation_queries = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "Techniques for improving retrieval-augmented generation systems.",
    "What was the final score of yesterday's football match?",  # fuera del corpus
]

qualitative_review = pd.DataFrame({
    "consulta": evaluation_queries,
    "corrección": [""] * len(evaluation_queries),
    "relevancia": [""] * len(evaluation_queries),
    "fidelidad_a_evidencias": [""] * len(evaluation_queries),
    "integra_varios_documentos": [""] * len(evaluation_queries),
    "abstención_adecuada": [""] * len(evaluation_queries),
    "observaciones": [""] * len(evaluation_queries),
})
display(qualitative_review)
# Después de completar los juicios manuales:
# qualitative_review.to_csv("evaluacion_cualitativa.csv", index=False)


## Conclusión

El sistema conecta todas las etapas de un RAG verificable: datos públicos reproducibles, representación densa, búsqueda persistente, re-ranking, generación restringida al corpus, citas y abstención explícita. La interfaz y el contenedor permiten que un evaluador formule consultas nuevas desde un navegador sin instalar dependencias.
